# Analyze your health data

Once a phone has synced, this notebook turns what is in your PulsHealth
database into charts and a few numbers you can act on: how active you have
been, where your resting heart rate and HRV are heading, how much and how
consistently you sleep, what your workouts look like month by month, and
whether any of those move together. The last section builds a compact
markdown summary of the same data — the shape of the product API's
`GET /v1/summary` — and shows how to hand it to an AI assistant.

It is **not** a tour of the tables: for the schema, the units and the
query patterns behind each frame, read
[`docs/database-guide.md`](../docs/database-guide.md). Everything here
reads the same daily surfaces the product API serves — `metric_daily` for
quantity types, `activity_summaries` for the rings, the sleep category
samples and `workouts` — never a raw hypertable scan.

## What you need

- **A database connection.** `DATABASE_URL` (a read-only role such as
  `grafana` is enough), or `PULS_DB_PASSWORD` / `POSTGRES_PASSWORD` in
  `.env` or `server/.env` — the notebook searches the working directory and
  its parents, so it works from the repository root or from `notebooks/`.
  Set `PULS_DB_SSH_HOST` to reach a remote host through an SSH tunnel.
  Keep secrets in `.env`; nothing in this file should ever hold a password.
- **The server's time zone.** Days in `metric_daily` are already cut by
  `PULS_TIME_ZONE` (the database's `puls_time_zone()`), which must match the
  phone. The notebook reads that setting for the raw-timestamp tables it
  buckets itself (sleep, workouts); set `PULS_ANALYSIS_TZ` to override.
- `PULS_LOOKBACK_DAYS` (default 90) sets the window, which ends on the last
  day that has data rather than today, so a database that stopped syncing
  still shows its last months. `PULS_USER_ID` picks a person on a shared
  server; by default the most recently updated user is analyzed.

```bash
python3 -m pip install -r notebooks/requirements.txt
jupyter lab notebooks/healthkit_database_exploration.ipynb
```

In [ ]:
import os
import socket
import subprocess
import time
from datetime import date, datetime, timedelta
from pathlib import Path
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sqlalchemy as sa
from IPython.display import Markdown, display
from sqlalchemy import text

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

LOOKBACK_DAYS = int(os.getenv("PULS_LOOKBACK_DAYS", "90"))


def note(message: str) -> None:
    # One-line "nothing to show" notes: a sparse database skips a plot, never raises.
    display(Markdown(f"*{message}*"))

In [ ]:
def normalize_database_url(url: str) -> str:
    if url.startswith("postgres://"):
        return "postgresql+psycopg://" + url[len("postgres://"):]
    if url.startswith("postgresql://"):
        return "postgresql+psycopg://" + url[len("postgresql://"):]
    return url


# Only connection settings are read from .env files. Anything else in a
# developer's .env (API keys included) stays out of the kernel, which is what
# keeps the optional Claude cell at the end off unless you export the key.
ENV_FILE_KEYS = ("DATABASE_URL", "POSTGRES_PASSWORD")
ENV_FILE_PREFIX = "PULS_"


def load_env_file(path: Path) -> dict[str, str]:
    loaded = {}
    if not path.exists():
        return loaded

    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('\"').strip("'")
        if not (key in ENV_FILE_KEYS or key.startswith(ENV_FILE_PREFIX)):
            continue
        if not os.getenv(key):
            os.environ[key] = value
            loaded[key] = value
    return loaded


def candidate_env_files() -> list[Path]:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base / ".env", base / "server" / ".env"))
    seen = set()
    unique_candidates = []
    for candidate in candidates:
        resolved = candidate.resolve(strict=False)
        if resolved not in seen:
            seen.add(resolved)
            unique_candidates.append(candidate)
    return unique_candidates


loaded_env_files = []
for env_file in candidate_env_files():
    loaded = load_env_file(env_file)
    if loaded:
        loaded_env_files.append(str(env_file))


def port_is_open(host: str, port: int, timeout: float = 0.4) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def ensure_ssh_tunnel() -> str | None:
    ssh_host = os.getenv("PULS_DB_SSH_HOST")
    if not ssh_host:
        return None

    local_host = os.getenv("PULS_DB_HOST", "127.0.0.1")
    local_port = int(os.getenv("PULS_DB_PORT", "15432"))
    remote_host = os.getenv("PULS_DB_SSH_REMOTE_HOST", "127.0.0.1")
    remote_port = int(os.getenv("PULS_DB_SSH_REMOTE_PORT", "5432"))

    if port_is_open(local_host, local_port):
        return f"existing SSH tunnel via {ssh_host}"

    subprocess.Popen(
        [
            "ssh",
            "-N",
            "-L",
            f"{local_port}:{remote_host}:{remote_port}",
            "-o",
            "BatchMode=yes",
            "-o",
            "ExitOnForwardFailure=yes",
            ssh_host,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(20):
        if port_is_open(local_host, local_port):
            return f"new SSH tunnel via {ssh_host}"
        time.sleep(0.25)
    raise RuntimeError(f"PULS_DB_SSH_HOST is set to {ssh_host}, but the SSH tunnel did not open.")


def database_url_from_env() -> tuple[str, str]:
    url = os.getenv("DATABASE_URL")
    if url:
        return normalize_database_url(url), "DATABASE_URL"

    tunnel_source = ensure_ssh_tunnel()
    host = os.getenv("PULS_DB_HOST", "127.0.0.1")
    port = os.getenv("PULS_DB_PORT", "5432")
    name = os.getenv("PULS_DB_NAME", "postgres")
    user = os.getenv("PULS_DB_USER", "postgres")
    password = os.getenv("PULS_DB_PASSWORD") or os.getenv("POSTGRES_PASSWORD")
    if not password:
        raise RuntimeError(
            "No database password found. Set DATABASE_URL, or add PULS_DB_PASSWORD "
            "or POSTGRES_PASSWORD to .env/server/.env."
        )

    source = tunnel_source or "env connection settings"
    return f"postgresql+psycopg://{quote_plus(user)}:{quote_plus(password)}@{host}:{port}/{name}", source


DATABASE_URL, CONNECTION_SOURCE = database_url_from_env()
engine = sa.create_engine(DATABASE_URL, pool_pre_ping=True)


def q(query: str, params: dict | None = None) -> pd.DataFrame:
    return pd.read_sql_query(text(query), engine, params=params or {})


with engine.connect() as conn:
    version = conn.execute(text("select version()")).scalar_one()
    server_time_zone = conn.execute(text("select puls_time_zone()")).scalar_one()

# The calendar every day-grain figure is cut in. metric_daily and the
# Activity rings are already bucketed server-side by PULS_TIME_ZONE; the
# same zone is applied here to the tables the notebook buckets itself.
ANALYSIS_TZ = os.getenv("PULS_ANALYSIS_TZ") or server_time_zone

source_note = f" Loaded env from `{', '.join(loaded_env_files)}`." if loaded_env_files else ""
display(Markdown(f"Connected via `{CONNECTION_SOURCE}`.{source_note}"))
display(Markdown(f"Server time zone `{server_time_zone}`; analyzing in `{ANALYSIS_TZ}` over the last {LOOKBACK_DAYS} days with data."))
print(version)

## 1. Coverage — what you have

Which types have a daily value, over how many days, and when the phone
last uploaded. `metric_daily` is the server's best daily figure per type
(the on-device HealthKit daily aggregate when the app synced one, else a
rollup of raw samples — the `source` column says which), so a type that
appears here is one the rest of the notebook can chart. The Activity
rings, sleep and workouts are counted separately because they are not
quantity types.

In [ ]:
users = q(
    '''
    SELECT u.id, u.name, u.updated_at,
           (SELECT max(received_at) FROM batches b WHERE b.user_id = u.id) AS last_sync
    FROM users u
    ORDER BY last_sync DESC NULLS LAST, u.updated_at DESC NULLS LAST, u.created_at
    '''
)
if users.empty:
    raise RuntimeError("The users table is empty; has the schema been applied?")

USER_ID = os.getenv("PULS_USER_ID") or str(users.iloc[0]["id"])
user_row = users[users["id"].astype(str) == USER_ID]
USER_NAME = user_row.iloc[0]["name"] if not user_row.empty else None
LAST_SYNC = user_row.iloc[0]["last_sync"] if not user_row.empty else None
display(Markdown(f"Analyzing user `{USER_ID}`" + (f" ({USER_NAME})" if USER_NAME else "") + f"; {len(users)} user(s) in the database."))

# The window ends on the last day anything was recorded for this user,
# so a database that stopped syncing still shows its last months.
bounds = q(
    '''
    SELECT greatest(
        (SELECT max(day) FROM metric_daily WHERE user_id = :user),
        (SELECT max(date) FROM activity_summaries WHERE user_id = :user),
        (SELECT max((cs.end_ts AT TIME ZONE :tz)::date) FROM category_samples cs
           JOIN sample_types st USING (type_id)
          WHERE cs.user_id = :user AND st.identifier = 'HKCategoryTypeIdentifierSleepAnalysis'),
        (SELECT max((start_ts AT TIME ZONE :tz)::date) FROM workouts WHERE user_id = :user)
    ) AS last_day
    ''',
    {"user": USER_ID, "tz": ANALYSIS_TZ},
)
last_day = bounds.iloc[0]["last_day"]
END_DAY = pd.Timestamp(last_day).date() if pd.notna(last_day) else date.today()
START_DAY = END_DAY - timedelta(days=LOOKBACK_DAYS - 1)
# Instants for the raw-timestamp tables: the window's first midnight and the
# midnight after its last day, in the analysis zone.
WINDOW = {
    "user": USER_ID, "start": START_DAY, "end": END_DAY, "tz": ANALYSIS_TZ,
    "start_instant": pd.Timestamp(START_DAY, tz=ANALYSIS_TZ).to_pydatetime(),
    "end_instant": pd.Timestamp(END_DAY + timedelta(days=1), tz=ANALYSIS_TZ).to_pydatetime(),
}
if pd.isna(last_day):
    note("No daily data for this user yet — every section below will skip. Sync a phone first.")
display(Markdown(f"Window: **{START_DAY} to {END_DAY}** ({LOOKBACK_DAYS} calendar days in `{ANALYSIS_TZ}`)."))

coverage = q(
    '''
    SELECT m.identifier, st.unit,
           count(*)::int AS days_with_data,
           min(m.day) AS first_day, max(m.day) AS last_day,
           sum(CASE WHEN m.source = 'aggregate' THEN 1 ELSE 0 END)::int AS days_from_device_aggregate
    FROM metric_daily m
    JOIN sample_types st USING (type_id)
    WHERE m.user_id = :user AND m.value IS NOT NULL
    GROUP BY m.identifier, st.unit
    ORDER BY days_with_data DESC, m.identifier
    ''',
    {"user": USER_ID},
)
display(Markdown("### Daily metrics (all time)"))
if coverage.empty:
    note("metric_daily has no rows for this user. It needs a daily sum/average aggregate series or raw samples of a type that has one configured.")
else:
    display(coverage)

extras = q(
    '''
    SELECT 'activity rings (days)' AS what, count(*)::int AS n, min(date)::text AS first, max(date)::text AS last
    FROM activity_summaries WHERE user_id = :user
    UNION ALL
    SELECT 'sleep samples', count(*)::int, min(cs.start_ts)::date::text, max(cs.end_ts)::date::text
    FROM category_samples cs JOIN sample_types st USING (type_id)
    WHERE cs.user_id = :user AND st.identifier = 'HKCategoryTypeIdentifierSleepAnalysis'
    UNION ALL
    SELECT 'workouts', count(*)::int, min(start_ts)::date::text, max(start_ts)::date::text
    FROM workouts WHERE user_id = :user
    ''',
    {"user": USER_ID},
)
display(Markdown("### Rings, sleep and workouts (all time)"))
display(extras)

## 2. Daily activity

Steps and active energy come from `metric_daily`; exercise minutes and
stand hours from the Activity rings when the phone synced them. Each
chart shows the day's value as bars and a trailing 7-day mean as a line,
which is the number to watch — single days swing with a long walk or a
rest day, the weekly mean does not. Days with no value are gaps, not
zeros: they are left out of every mean in this notebook.

In [ ]:
STEPS = "HKQuantityTypeIdentifierStepCount"
ACTIVE_ENERGY = "HKQuantityTypeIdentifierActiveEnergyBurned"
EXERCISE_TIME = "HKQuantityTypeIdentifierAppleExerciseTime"
RESTING_HR = "HKQuantityTypeIdentifierRestingHeartRate"
HRV_SDNN = "HKQuantityTypeIdentifierHeartRateVariabilitySDNN"
BODY_MASS = "HKQuantityTypeIdentifierBodyMass"
BODY_FAT = "HKQuantityTypeIdentifierBodyFatPercentage"


def daily_metric(identifier: str) -> pd.Series:
    # One value per calendar day (already cut by PULS_TIME_ZONE) within the window,
    # indexed by day. Empty when the type has no daily figure.
    df = q(
        '''
        SELECT day, value
        FROM metric_daily
        WHERE user_id = :user AND identifier = :identifier
          AND day BETWEEN :start AND :end AND value IS NOT NULL
        ORDER BY day
        ''',
        {**WINDOW, "identifier": identifier},
    )
    series = pd.Series(df["value"].astype(float).values, index=pd.to_datetime(df["day"]), name=identifier)
    return series


def rolling_week(series: pd.Series) -> pd.Series:
    # Trailing 7 calendar days (not 7 rows), so gaps do not stretch the window.
    return series.sort_index().rolling("7D", min_periods=1).mean()


def plot_daily(series: pd.Series, title: str, ylabel: str, decimals: int = 0) -> None:
    if series.empty:
        note(f"No data for {title.lower()} in this window — skipping the chart.")
        return
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.bar(series.index, series.values, width=0.8, color="#b8c4d6", label="daily")
    ax.plot(rolling_week(series).index, rolling_week(series).values, color="#1f4e79", linewidth=2, label="7-day mean")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.legend(loc="upper left", frameon=False)
    ax.set_xlim(pd.Timestamp(START_DAY) - pd.Timedelta(days=1), pd.Timestamp(END_DAY) + pd.Timedelta(days=1))
    plt.tight_layout()
    plt.show()
    display(Markdown(
        f"{len(series)} days with data; mean **{series.mean():,.{decimals}f}**, "
        f"lowest day {series.min():,.{decimals}f}, highest day {series.max():,.{decimals}f}."
    ))


rings = q(
    '''
    SELECT date AS day, move_kcal, move_goal_kcal, exercise_min, exercise_goal_min,
           stand_hours, stand_goal_hours
    FROM activity_summaries
    WHERE user_id = :user AND date BETWEEN :start AND :end
    ORDER BY date
    ''',
    WINDOW,
)
rings.index = pd.to_datetime(rings.pop("day"))

steps = daily_metric(STEPS)
active_energy = daily_metric(ACTIVE_ENERGY)
if not rings.empty and rings["exercise_min"].notna().any():
    exercise = rings["exercise_min"].dropna().astype(float)
    exercise_source = "the Activity rings"
else:
    exercise = daily_metric(EXERCISE_TIME)
    exercise_source = "metric_daily"

plot_daily(steps, "Steps", "steps")
plot_daily(active_energy, "Active energy", "kcal")
plot_daily(exercise, f"Exercise minutes (from {exercise_source})", "min")

if not rings.empty:
    closed = pd.DataFrame({
        "move": rings["move_kcal"] >= rings["move_goal_kcal"],
        "exercise": rings["exercise_min"] >= rings["exercise_goal_min"],
        "stand": rings["stand_hours"] >= rings["stand_goal_hours"],
    })
    display(Markdown("Rings closed on this many days: " + ", ".join(
        f"**{name}** {int(col.sum())}/{int(col.notna().sum())}" for name, col in closed.items()
    )))

## 3. Resting heart rate and HRV

Both are one reading a day from the Watch, so `metric_daily` holds the
day's value directly. Resting heart rate drifting up over a couple of
weeks, or HRV (SDNN) drifting down, is the usual early sign of
overtraining, illness or poor sleep — which is why the 7-day line matters
more than any single morning. HRV in particular is noisy day to day and
only comparable with yourself.

In [ ]:
resting_hr = daily_metric(RESTING_HR)
hrv = daily_metric(HRV_SDNN)

plot_daily(resting_hr, "Resting heart rate", "bpm")
plot_daily(hrv, "Heart rate variability (SDNN)", "ms")

if not resting_hr.empty and len(resting_hr) >= 14:
    recent = rolling_week(resting_hr).iloc[-1]
    baseline = resting_hr.iloc[:-7].mean() if len(resting_hr) > 7 else resting_hr.mean()
    display(Markdown(
        f"Last 7-day mean resting HR **{recent:.0f} bpm** against a window baseline of "
        f"**{baseline:.0f} bpm** ({recent - baseline:+.1f})."
    ))

## 4. Sleep duration and consistency

Sleep is a category type: each row is one stage (in bed, core, deep,
REM, awake) with a start and end, and the Watch and a third-party sleep
app can both record the same night. This cell keeps only the asleep
stages, assigns each sample to the day you woke up (its end, in the
analysis zone), sums per source and per wake-up day and keeps the source
that recorded the most sleep — close to what `GET /v1/summary` does
with its longest-session rule. Consistency is the spread of both how
long you slept and when you went to bed; a regular bedtime tends to
predict good sleep better than the occasional long night.

In [ ]:
sleep_rows = q(
    '''
    SELECT cs.start_ts, cs.end_ts, cs.value, COALESCE(cl.label, cs.value::text) AS stage,
           COALESCE(cs.source_id, 0) AS source_id
    FROM category_samples cs
    JOIN sample_types st USING (type_id)
    LEFT JOIN category_labels cl ON cl.type_identifier = st.identifier AND cl.value = cs.value
    WHERE cs.user_id = :user
      AND st.identifier = 'HKCategoryTypeIdentifierSleepAnalysis'
      AND cs.end_ts >= :start_instant - interval '1 day'
      AND cs.start_ts < :end_instant + interval '1 day'
    ORDER BY cs.start_ts
    ''',
    WINDOW,
)

ASLEEP_VALUES = {1, 3, 4, 5}  # asleepUnspecified, core, deep, REM — not inBed (0) or awake (2)
nights = pd.DataFrame(columns=["wake_day", "asleep_min", "bedtime_h"])
if sleep_rows.empty:
    note("No sleep samples in this window — skipping.")
else:
    asleep = sleep_rows[sleep_rows["value"].isin(ASLEEP_VALUES)].copy()
    if asleep.empty:
        note("Sleep samples exist but none are asleep stages (only in-bed/awake) — skipping.")
    else:
        asleep["start_local"] = asleep["start_ts"].dt.tz_convert(ANALYSIS_TZ)
        asleep["end_local"] = asleep["end_ts"].dt.tz_convert(ANALYSIS_TZ)
        asleep["wake_day"] = asleep["end_local"].dt.normalize().dt.tz_localize(None)
        asleep["minutes"] = (asleep["end_ts"] - asleep["start_ts"]).dt.total_seconds() / 60
        per_source = (
            asleep.groupby(["wake_day", "source_id"])
            .agg(asleep_min=("minutes", "sum"), first_start=("start_local", "min"))
            .reset_index()
        )
        best = per_source.sort_values("asleep_min", ascending=False).drop_duplicates("wake_day")
        best = best[(best["wake_day"] >= pd.Timestamp(START_DAY)) & (best["wake_day"] <= pd.Timestamp(END_DAY))]
        # Bedtime as hours after 18:00, so 23:30 -> 5.5 and 01:00 -> 7.0 sort correctly.
        best["bedtime_h"] = ((best["first_start"].dt.hour + best["first_start"].dt.minute / 60) - 18) % 24
        nights = best[["wake_day", "asleep_min", "bedtime_h"]].sort_values("wake_day").reset_index(drop=True)

        stage_minutes = asleep.groupby("stage")["minutes"].sum().sort_values(ascending=False)
        display(Markdown("Asleep minutes by stage over the window: " + ", ".join(
            f"{stage} {minutes:,.0f}" for stage, minutes in stage_minutes.items()
        )))

if nights.empty:
    if not sleep_rows.empty:
        note("No full nights inside the window — skipping the chart.")
else:
    hours = pd.Series(nights["asleep_min"].values / 60, index=nights["wake_day"])
    plot_daily(hours, "Sleep per night (by wake-up day)", "hours asleep", decimals=1)
    bedtimes = nights["bedtime_h"]
    typical_bedtime = (18 + bedtimes.median()) % 24
    display(Markdown(
        f"{len(nights)} nights: mean **{hours.mean():.1f} h**, night-to-night spread (std) "
        f"**{hours.std():.1f} h**; typical bedtime **{int(typical_bedtime):02d}:{int((typical_bedtime % 1) * 60):02d}** "
        f"with a spread of **{bedtimes.std() * 60:.0f} min**."
    ))

## 5. Workouts by type and month

Every workout is one row with its HealthKit activity type, duration,
energy and distance. Months are cut in the analysis zone from the start
time. The count per month is the honest measure of consistency; minutes
and distance show where the time went.

In [ ]:
workouts = q(
    '''
    SELECT start_ts, end_ts, activity_type,
           duration_s / 60.0 AS duration_min, energy_kcal, distance_m
    FROM workouts
    WHERE user_id = :user
      AND start_ts >= :start_instant
      AND start_ts < :end_instant
    ORDER BY start_ts
    ''',
    WINDOW,
)

if workouts.empty:
    note("No workouts in this window — skipping.")
else:
    workouts["month"] = workouts["start_ts"].dt.tz_convert(ANALYSIS_TZ).dt.strftime("%Y-%m")
    by_type = (
        workouts.groupby("activity_type")
        .agg(count=("activity_type", "size"), minutes=("duration_min", "sum"),
             kcal=("energy_kcal", "sum"), km=("distance_m", lambda m: m.sum() / 1000))
        .sort_values("count", ascending=False)
        .round(1)
    )
    display(by_type)

    by_month = workouts.pivot_table(index="month", columns="activity_type", values="start_ts", aggfunc="size", fill_value=0)
    ax = by_month.plot.bar(stacked=True, figsize=(10, 4), width=0.7, rot=0)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_title("Workouts per month by activity type")
    ax.set_ylabel("workouts")
    ax.set_xlabel("")
    ax.legend(frameon=False, title=None)
    plt.tight_layout()
    plt.show()

## 6. Correlations you can actually read

Three pairs where a relationship is plausible and the direction is easy
to interpret: last night's sleep against today's resting heart rate and
HRV (does a short night show up in the morning?), and steps against active
energy (a sanity check — these should track closely, and a weak
correlation usually means a source-mixing problem, not physiology).

**Read the `n` before the `r`.** A Pearson r over a dozen days means
nothing; the cell refuses to draw a scatter below 14 paired days, and
even at 90 days a single illness or holiday can create or hide a
correlation. This is your own data, not a study — treat it as a prompt
for questions, not an answer.

In [ ]:
sleep_by_day = (
    pd.Series(nights["asleep_min"].values / 60, index=pd.DatetimeIndex(nights["wake_day"]), name="sleep_h")
    if not nights.empty else pd.Series(dtype=float, name="sleep_h")
)

pairs = [
    ("Sleep (h) vs resting HR (bpm) the same day", sleep_by_day, resting_hr.rename("resting_hr")),
    ("Sleep (h) vs HRV SDNN (ms) the same day", sleep_by_day, hrv.rename("hrv_ms")),
    ("Steps vs active energy (kcal)", steps.rename("steps"), active_energy.rename("active_kcal")),
]
MIN_PAIRED_DAYS = 14

results = []
for title, x, y in pairs:
    joined = pd.concat([x, y], axis=1, join="inner").dropna()
    n = len(joined)
    r = joined.iloc[:, 0].corr(joined.iloc[:, 1]) if n >= 3 else float("nan")
    results.append({"pair": title, "paired_days": n, "pearson_r": round(r, 2) if pd.notna(r) else None})
    if n < MIN_PAIRED_DAYS:
        note(f"{title}: only {n} paired days (need {MIN_PAIRED_DAYS}) — no scatter.")
        continue
    ax = sns.regplot(data=joined, x=joined.columns[0], y=joined.columns[1], scatter_kws={"s": 18, "alpha": 0.7}, line_kws={"color": "#1f4e79"})
    ax.set_title(f"{title}  (n = {n}, r = {r:.2f})")
    plt.tight_layout()
    plt.show()

display(pd.DataFrame(results))

## 7. Hand it to an assistant

An assistant does not need the frames above; it needs the few dozen
numbers that describe them. This cell renders the window as the same
markdown page the product API serves at `GET /v1/summary` — the same
section names (Activity, Heart, Sleep, Workouts, Body, Coverage), the
same deduplicated daily figures, the same caveat line — so whatever you
build here reads the way the API's page does, and a chat that has seen
one understands the other.

In [ ]:
def fmt_number(value: float, decimals: int = 0) -> str:
    return f"{value:,.{decimals}f}"


def fmt_minutes(minutes: float) -> str:
    total = int(round(minutes))
    h, m = divmod(total, 60)
    if h == 0:
        return f"{m} min"
    if m == 0:
        return f"{h} h"
    return f"{h} h {m:02d} min"


def days_with(n: int) -> str:
    return f"{n} {'day' if n == 1 else 'days'} with data"


def latest_reading(identifier: str):
    df = q(
        '''
        SELECT qs.value, st.unit, qs.start_ts
        FROM quantity_samples qs
        JOIN sample_types st USING (type_id)
        WHERE qs.user_id = :user AND st.identifier = :identifier
        ORDER BY qs.start_ts DESC
        LIMIT 1
        ''',
        {"user": USER_ID, "identifier": identifier},
    )
    return None if df.empty else df.iloc[0]


def build_summary_markdown() -> str:
    days = LOOKBACK_DAYS
    lines = []
    title = f"# Health summary — last {days} days"
    if USER_NAME and str(USER_NAME).strip():
        title = f"# Health summary for {str(USER_NAME).strip()} — last {days} days"
    lines.append(title)
    lines.append("")
    lines.append(
        f"{START_DAY} to {END_DAY}, {days} calendar days in {ANALYSIS_TZ} (the server's time zone). "
        f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M')}."
    )
    covered = set()

    def cover(series: pd.Series) -> None:
        covered.update(pd.DatetimeIndex(series.index).date)

    activity = []
    if not steps.empty:
        cover(steps)
        activity.append(f"- Steps: {fmt_number(steps.mean())} per day on average ({fmt_number(steps.sum())} in total; {days_with(len(steps))})")
    if not active_energy.empty:
        cover(active_energy)
        activity.append(f"- Active energy: {fmt_number(active_energy.mean())} kcal per day on average ({fmt_number(active_energy.sum())} kcal in total; {days_with(len(active_energy))})")
    if not exercise.empty:
        cover(exercise)
        source_note = ", from the Activity rings" if exercise_source == "the Activity rings" else ""
        activity.append(f"- Exercise: {fmt_number(exercise.mean())} min per day on average ({fmt_number(exercise.sum())} min in total; {days_with(len(exercise))}{source_note})")
    if not rings.empty and rings["stand_hours"].notna().any():
        stand = rings["stand_hours"].dropna()
        cover(stand)
        activity.append(f"- Stand: {fmt_number(stand.mean(), 1)} hours per day on average (fewest {fmt_number(stand.min())}, most {fmt_number(stand.max())}; {days_with(len(stand))}, from the Activity rings)")
    if activity:
        lines += ["", "## Activity", *activity]

    heart = []
    if not resting_hr.empty:
        cover(resting_hr)
        heart.append(f"- Resting heart rate: {fmt_number(resting_hr.mean())} bpm on average (lowest day {fmt_number(resting_hr.min())}, highest day {fmt_number(resting_hr.max())}; {days_with(len(resting_hr))})")
    if not hrv.empty:
        cover(hrv)
        heart.append(f"- Heart rate variability (SDNN): {fmt_number(hrv.mean())} ms on average ({days_with(len(hrv))})")
    if heart:
        lines += ["", "## Heart", *heart]

    if not nights.empty:
        covered.update(pd.DatetimeIndex(nights["wake_day"]).date)
        n = len(nights)
        lines += ["", "## Sleep",
                  f"- Asleep: {fmt_minutes(nights['asleep_min'].mean())} per night on average "
                  f"(shortest {fmt_minutes(nights['asleep_min'].min())}, longest {fmt_minutes(nights['asleep_min'].max())}; "
                  f"{n} {'night' if n == 1 else 'nights'} with data)"]

    if not workouts.empty:
        covered.update(workouts["start_ts"].dt.tz_convert(ANALYSIS_TZ).dt.date)
        count = len(workouts)
        line = f"- {count} {'workout' if count == 1 else 'workouts'}, {fmt_minutes(workouts['duration_min'].fillna(0).sum())} in total"
        if workouts["distance_m"].notna().any():
            line += f", {fmt_number(workouts['distance_m'].fillna(0).sum() / 1000, 1)} km"
        top = workouts["activity_type"].value_counts().head(3)
        lines += ["", "## Workouts", line,
                  "- Most frequent: " + ", ".join(f"{t} ({c})" for t, c in top.items())]

    body = []
    weight = latest_reading(BODY_MASS)
    if weight is not None:
        body.append(f"- Weight: {fmt_number(weight['value'], 1)} {weight['unit']} (latest reading, {weight['start_ts'].tz_convert(ANALYSIS_TZ).strftime('%Y-%m-%d')})")
    fat = latest_reading(BODY_FAT)
    if fat is not None:
        body.append(f"- Body fat: {fmt_number(fat['value'] * 100, 1)} % (latest reading, {fat['start_ts'].tz_convert(ANALYSIS_TZ).strftime('%Y-%m-%d')})")
    if body:
        lines += ["", "## Body", *body]

    last_sync = pd.Timestamp(LAST_SYNC).tz_convert(ANALYSIS_TZ).strftime("%Y-%m-%d %H:%M") if pd.notna(LAST_SYNC) else "never"
    in_window = {d for d in covered if START_DAY <= d <= END_DAY}
    lines += ["", "## Coverage"]
    if not in_window:
        lines.append(f"- No data in this range. Last sync: {last_sync}.")
    else:
        lines.append(f"- Last sync: {last_sync}. {len(in_window)} of {days} days have data.")
        lines.append("- Daily figures are HealthKit's deduplicated daily values (iPhone and Watch overlap already removed), "
                     "never sums of raw samples; each workout is counted once. Days without data are left out of the averages, not counted as zero.")
    return "\n".join(lines) + "\n"


summary_markdown = build_summary_markdown()
print(summary_markdown)

### Three ways to give an assistant this picture

1. **Paste it.** Copy the page above into any chat — Claude, ChatGPT, a
   local model — and ask your question underneath. Nothing else leaves your
   machine; the numbers are averages and totals, not samples.
2. **Ask the API for it.** The running product API renders the same page
   with one request, no notebook needed (`docs/ai.md`, "No MCP at all"):
   ```bash
   curl -H "Authorization: Bearer $PULS_API_TOKEN" "$API/v1/summary?range=30d"
   ```
   `range` is `7d`, `14d`, `30d` or `90d`; `format=json` returns the numbers.
3. **Connect the MCP server.** `server/mcp` gives Claude Desktop, Claude
   Code, Cursor or a remote connector read-only tools over the API —
   `get_summary` for this page, `get_daily_metrics`, `get_sleep`,
   `list_workouts` and the rest for follow-up questions — so the assistant
   can pull what it needs instead of what you pasted. Setup is in
   [`docs/ai.md`](../docs/ai.md).

The cell below is the first option automated: it sends the summary and a
question to Claude through the `anthropic` SDK. It is **skipped unless
`ANTHROPIC_API_KEY` is exported in the kernel's environment** (the `.env`
loader above reads connection settings only, so a key in a `.env` file does
not count), and the SDK is deliberately not in `notebooks/requirements.txt`
(`pip install anthropic` to use it), so the notebook never makes a network
call by default and CI never does. Nothing but the markdown above and your
question is sent.

In [ ]:
QUESTION = (
    "What stands out in this data, and what one change would you suggest I try for the next two weeks? "
    "Be specific about which figures you are basing that on."
)
SYSTEM = (
    "You are looking at a person's own Apple Health data, summarized as daily averages and totals over one window. "
    "Reason only from the figures given, say plainly when the coverage is too thin to conclude anything, "
    "and do not give medical diagnoses."
)

if not os.getenv("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY is not set — skipping the API call. Paste the summary above into any chat instead.")
else:
    try:
        import anthropic
    except ImportError:
        print("The anthropic SDK is not installed: `pip install anthropic` (it is intentionally not in notebooks/requirements.txt).")
    else:
        client = anthropic.Anthropic()
        try:
            response = client.messages.create(
                model="claude-sonnet-5",
                max_tokens=4096,
                system=SYSTEM,
                messages=[{"role": "user", "content": f"{summary_markdown}\n\n{QUESTION}"}],
            )
        except anthropic.AuthenticationError:
            print("Invalid API key.")
        except anthropic.RateLimitError as exc:
            print(f"Rate limited; retry after {exc.response.headers.get('retry-after', '60')} s.")
        except anthropic.APIStatusError as exc:
            print(f"API error {exc.status_code}: {exc.message}")
        except anthropic.APIConnectionError:
            print("Network error reaching the API.")
        else:
            if response.stop_reason == "refusal":
                print("The model declined to answer this request.")
            else:
                display(Markdown("\n\n".join(block.text for block in response.content if block.type == "text")))

## Going further

`q()` runs any SQL as a frame, and `daily_metric()` returns any type's
daily series — the identifiers are in
[`docs/protocol/catalog.json`](../docs/protocol/catalog.json). For raw
samples, series and routes, [`docs/database-guide.md`](../docs/database-guide.md)
has the query patterns and the gotchas (compressed chunks, source
double-counting, the time-zone rule); for a file to take elsewhere, the
API's `GET /v1/export` and the `puls-export` CLI stream CSV or JSONL.